In [1]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# =========================
# 2. LOAD DATA
# =========================
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print(train_df.shape)
print(train_df.head())

# =========================
# 3. FEATURE ENGINEERING
# =========================

def feature_engineering(df):
    df = df.copy()
    
    # Split Cabin -> Deck / Num / Side
    df[["Deck", "CabinNum", "Side"]] = df["Cabin"].str.split("/", expand=True)
    
    # Group from PassengerId
    df["Group"] = df["PassengerId"].apply(lambda x: x.split("_")[0])
    
    # Family size
    df["FamilySize"] = df.groupby("Group")["Group"].transform("count")
    
    # Total spending
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    df["TotalSpend"] = df[spend_cols].sum(axis=1)
    
    # Drop useless columns
    df.drop(["Cabin", "Name", "PassengerId"], axis=1, inplace=True)
    
    return df

train_df = feature_engineering(train_df)
test_df = feature_engineering(test_df)

# =========================
# 4. SPLIT FEATURES/TARGET
# =========================
X = train_df.drop("Transported", axis=1)
y = train_df["Transported"]

# =========================
# 5. COLUMN TYPES
# =========================
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "bool"]).columns

# =========================
# 6. PREPROCESSING
# =========================

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# =========================
# 7. MODEL PIPELINE
# =========================
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

# =========================
# 8. TRAIN
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

# =========================
# 9. VALIDATION
# =========================
y_pred = model.predict(X_val)
acc = accuracy_score(y_val, y_pred)

print("Validation Accuracy:", acc)

# =========================
# 10. TEST PREDICTION
# =========================
test_pred = model.predict(test_df)

# =========================
# 11. SUBMISSION FILE
# =========================
submission = pd.read_csv("sample_submission.csv")
submission["Transported"] = test_pred

submission.to_csv("submission.csv", index=False)

print("Submission file created!")

(8693, 14)
  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4       

C:\Users\sharm\AppData\Local\Temp\ipykernel_25596\58514907.py:62: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "bool"]).columns


Validation Accuracy: 0.7418056354226567
Submission file created!
